# 01.01 · Python for AI engineers

**Chapter:** Python & Math for AI · **Milestone:** M1

## Goals
- [ ] Use type hints, dataclasses, comprehensions fluently
- [ ] Use generators (how streaming LLM responses work)
- [ ] Write async code with a concurrency limit (LLM calls are I/O bound)
- [ ] Import shared code from the repo's `aieng` package

## Resources
- [Python tutorial](https://docs.python.org/3/tutorial/) — free
- [CS50P](https://cs50.harvard.edu/python/) — free
- [Real Python – Async IO walkthrough](https://realpython.com/async-io-python/) — free

You don't need *all* of Python. AI engineering code is mostly: data shapes (dicts, lists, dataclasses / Pydantic),
iteration (comprehensions, generators), files and JSON, HTTP calls, and concurrency. This lesson covers exactly that.

## 1. Type hints and dataclasses
Type hints don't change runtime behaviour, but editors and tools (and LLM structured-output libraries like Pydantic, chapter 05)
read them. A `dataclass` gives you a typed record without boilerplate.

In [1]:
from dataclasses import dataclass, field, asdict


@dataclass
class Message:
    role: str  # "system" | "user" | "assistant"
    content: str
    tokens: int = 0
    tags: list[str] = field(default_factory=list)  # never use a mutable default directly


conversation = [
    Message("system", "You are terse."),
    Message("user", "What is RAG?", tokens=4),
]
conversation.append(Message("assistant", "Retrieval-augmented generation.", tokens=5, tags=["definition"]))
print(conversation[-1])
print(asdict(conversation[-1]))  # dict form: what an API wants as JSON

Message(role='assistant', content='Retrieval-augmented generation.', tokens=5, tags=['definition'])
{'role': 'assistant', 'content': 'Retrieval-augmented generation.', 'tokens': 5, 'tags': ['definition']}


## 2. Comprehensions: the everyday data tool
List, dict and set comprehensions replace most `for` loops that build collections.

In [2]:
messages_for_api = [{"role": m.role, "content": m.content} for m in conversation]
tokens_by_role = {m.role: m.tokens for m in conversation}
total_tokens = sum(m.tokens for m in conversation)  # generator expression: no intermediate list
user_texts = [m.content for m in conversation if m.role == "user"]

print(messages_for_api, tokens_by_role, total_tokens, user_texts, sep="\n")

[{'role': 'system', 'content': 'You are terse.'}, {'role': 'user', 'content': 'What is RAG?'}, {'role': 'assistant', 'content': 'Retrieval-augmented generation.'}]
{'system': 0, 'user': 4, 'assistant': 5}
9
['What is RAG?']


## 3. Generators: how streaming works
A generator `yield`s values one at a time. LLM SDKs return a generator of chunks when you pass `stream=True`;
you print each chunk as it arrives instead of waiting for the full answer.

In [4]:
import time


def fake_stream(text: str, delay: float = 0.02):
    """Pretend to be an LLM streaming one word at a time."""
    for word in text.split():
        time.sleep(delay)
        yield word + " "


for chunk in fake_stream("Tokens arrive one by one, so the user sees progress immediately."):
    print(chunk, end="", flush=True)
print()

Tokens arrive one by one, so the user sees progress immediately. 


## 4. Files, paths and JSON / JSONL
`pathlib.Path` for paths. JSON Lines (one JSON object per line) is the standard format for datasets, eval sets and logs:
you can append to it and stream it without loading everything.

In [5]:
import json
from aieng import DATA  # repo's shared package: DATA is the git-ignored data/ folder

log_file = DATA / "01-01-conversation.jsonl"
with log_file.open("w") as f:
    for m in conversation:
        f.write(json.dumps(asdict(m)) + "\n")

loaded = [Message(**json.loads(line)) for line in log_file.read_text().splitlines()]
assert loaded == conversation
print(log_file, "->", len(loaded), "messages round-tripped")

/Users/sebastian/Developer/ai-eng/data/01-01-conversation.jsonl -> 3 messages round-tripped


## 5. Async: doing many slow calls at once
An LLM call spends ~all its time **waiting on the network**. Running 100 calls one after another wastes that time.
`asyncio` lets one thread wait on many calls concurrently.

Two rules you'll use constantly:
- `asyncio.gather(*coros)` runs coroutines concurrently and returns results in order.
- `asyncio.Semaphore(n)` caps concurrency so you don't blow through provider rate limits.

In [7]:
import asyncio
import random


async def fake_llm_call(prompt: str) -> str:
    await asyncio.sleep(random.uniform(0.2, 0.4))  # network wait
    return prompt.upper()


prompts = [f"prompt {i}" for i in range(20)]

start = time.perf_counter()
for p in prompts[:5]:
    await fake_llm_call(p)  # notebooks allow top-level await
print(f"sequential, 5 calls: {time.perf_counter() - start:.2f}s")

start = time.perf_counter()
results = await asyncio.gather(*(fake_llm_call(p) for p in prompts))
print(f"concurrent, 20 calls: {time.perf_counter() - start:.2f}s")

sequential, 5 calls: 1.76s
concurrent, 20 calls: 0.39s


In [10]:
async def limited(prompts: list[str], max_concurrency: int = 5) -> list[str]:
    sem = asyncio.Semaphore(max_concurrency)

    async def one(p):
        async with sem:  # at most max_concurrency calls in flight
            return await fake_llm_call(p)

    return await asyncio.gather(*(one(p) for p in prompts))


start = time.perf_counter()
results = await limited(prompts, max_concurrency=5)
print(f"20 calls, max 5 at a time: {time.perf_counter() - start:.2f}s  (≈ 4 waves)")
assert results == [p.upper() for p in prompts]  # gather keeps order

20 calls, max 5 at a time: 1.28s  (≈ 4 waves)


## 6. Errors you must handle
Network calls fail: timeouts, rate limits (HTTP 429), server errors (5xx). Catch specific exceptions, retry the
retryable ones with backoff (chapter 05.05 does this properly), and let real bugs crash.

In [18]:
async def flaky_call(prompt: str) -> str:
    if random.random() < 0.3:
        raise TimeoutError("provider timed out")
    return await fake_llm_call(prompt)


async def with_retry(prompt: str, attempts: int = 4) -> str:
    for i in range(attempts):
        try:
            return await flaky_call(prompt)
        except TimeoutError:
            if i == attempts - 1:
                raise
            await asyncio.sleep(0.05 * 2**i)  # exponential backoff


random.seed(0)
print(await asyncio.gather(*(with_retry(p) for p in prompts[:5])))

['PROMPT 0', 'PROMPT 1', 'PROMPT 2', 'PROMPT 3', 'PROMPT 4']


## Exercises
1. Fetch these 6 URLs **sequentially** and then **concurrently** with `httpx.AsyncClient` + `asyncio.gather`; compare timings.
2. Add a `Semaphore(2)` and confirm the timing changes as expected.
3. Write a generator `batched(items, n)` that yields lists of size `n` (then check: Python 3.12 has `itertools.batched`).

In [26]:
import httpx

URLS = [
    "https://www.python.org",
    "https://huggingface.co",
    "https://pytorch.org",
    "https://numpy.org",
    "https://docs.pola.rs",
    "https://www.rust-lang.org",
]

# Exercise 1:
async with httpx.AsyncClient(timeout=20, follow_redirects=True) as http:
    start = time.perf_counter()
    for url in URLS:
        response = await http.get(url)
    print(f"sequential, 5 calls: {time.perf_counter() - start:.2f}s")

async with httpx.AsyncClient(timeout=20, follow_redirects=True) as http:
    start = time.perf_counter()
    results = await asyncio.gather(*(http.get(url) for url in URLS))
    print(f"concurrent, 5 calls: {time.perf_counter() - start:.2f}s")

# Exercise 2:
async with httpx.AsyncClient(timeout=20, follow_redirects=True) as http:
    sem = asyncio.Semaphore(2)
    async def one(url):
        async with sem:
            return await http.get(url)
    start = time.perf_counter()
    results = await asyncio.gather(*(one(url) for url in URLS))
    print(f"concurrent, 5 calls: {time.perf_counter() - start:.2f}s")


sequential, 5 calls: 1.77s
concurrent, 5 calls: 0.30s
concurrent, 5 calls: 0.33s


In [27]:
# Exercise 3:
def batched(items, n):
    for i in range(0, len(items), n):
        yield list(items[i:i+n])

assert list(batched(range(5), 2)) == [[0, 1], [2, 3], [4]]